In [8]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt

In [7]:
import pandas as pd
import os

# Define the filenames mapped to your generations
files_map = {
    'A52': 'a52samsungcleanedytcomment.csv',
    'A53': 'a53samsungcleanedytcomment.csv',
    'A54': 'a54samsungcleanedytcomment.csv',
    'A55': 'a55samsungcleanedytcomment.csv',
    'A56': 'a56samsungcleanedytcomment.csv'
}
DATA_DIR = "../data/processed/"

table_rows = []

# Loop through each generation to calculate the metrics
for idx, (gen, file_name) in enumerate(files_map.items(), 1):
    file_path = os.path.join(DATA_DIR, file_name)
    
    # 1. Raw Scraped Count (Assuming raw data was cleaned but total row count matches)
    # If you have separate raw files before text preprocessing, read those instead.
    # Otherwise, we use the processed baseline count.
    try:
        df_raw = pd.read_csv(file_path)
        total_raw = len(df_raw) # Substitute with raw counter if available
        total_cleaned = df_raw['clean_comment'].dropna().count()
    except Exception:
        total_raw, total_cleaned = 0, 0
        
    # 2. Extract from your final model dataframe (df is your active dataframe from the inference loop)
    # Filter by the 'generation' label column we added
    df_gen = df[df['generation'] == gen]
    
    aspect_matched = len(df_gen)
    
    # Final sentiments extracted (valid non-null prediction clauses)
    final_sentiments = df_gen['sentiment'].dropna().count()
    
    table_rows.append({
        'Generation': f"Generation {idx} ($G_{idx}$ - {gen})",
        'Raw Scraped Comments': total_raw,
        'Post-Preprocessing Cleaned': total_cleaned,
        'Aspect-Matched Clauses': aspect_matched,
        'Final Sentiments Extracted': final_sentiments
    })

# Convert to final DataFrame for visualization
metrics_df = pd.DataFrame(table_rows)

# Calculate totals row
totals = pd.Series({
    'Generation': 'Total Corpus',
    'Raw Scraped Comments': metrics_df['Raw Scraped Comments'].sum(),
    'Post-Preprocessing Cleaned': metrics_df['Post-Preprocessing Cleaned'].sum(),
    'Aspect-Matched Clauses': metrics_df['Aspect-Matched Clauses'].sum(),
    'Final Sentiments Extracted': metrics_df['Final Sentiments Extracted'].sum()
})

metrics_df = pd.concat([metrics_df, pd.DataFrame([totals])], ignore_index=True)
print("📊 Generated Table I Data for your IEEE Proceeding:")
display(metrics_df)

📊 Generated Table I Data for your IEEE Proceeding:


,Generation,Raw Scraped Comments,Post-Preprocessing Cleaned,Aspect-Matched Clauses,Final Sentiments Extracted
0,Generation 1 ($G_1$ - A52),18900,18859,7913,7913
1,Generation 2 ($G_2$ - A53),10450,10450,5127,5127
2,Generation 3 ($G_3$ - A54),9526,9526,4396,4396
3,Generation 4 ($G_4$ - A55),16133,16133,7990,7990
4,Generation 5 ($G_5$ - A56),6998,6997,3510,3510
5,Total Corpus,62007,61965,28936,28936


In [13]:
# Load the inference dataset
input_csv = "../data/final_results/final_generational_sentiment.csv"
try:
    df_final = pd.read_csv(input_csv)
except FileNotFoundError:
    df_final = df.copy() # Fallback to active dataframe

# Define order and target domains
generations = ['A52', 'A53', 'A54', 'A55', 'A56']
x_ticks = ['$G_1$\n(A52)', '$G_2$\n(A53)', '$G_3$\n(A54)', '$G_4$\n(A55)', '$G_5$\n(A56)']
target_aspects = ['Performance', 'Camera', 'Battery', 'Display']

# ---------------------------------------------------------------------
# DATA AGGREGATION ENGINE (Exploding comma-separated aspects)
# ---------------------------------------------------------------------
records = []
for _, row in df_final.iterrows():
    gen = row['generation']
    snt = str(row['sentiment']).lower().strip()
    aspects_str = str(row['detected_aspects']).replace('[','').replace(']','').replace("'", "")
    
    for aspect in [a.strip() for a in aspects_str.split(',') if a.strip()]:
        if aspect in target_aspects:
            records.append({'Generation': gen, 'Aspect': aspect, 'Sentiment': snt})

agg_df = pd.DataFrame(records)

# Create Output Path
os.makedirs("../data/plots", exist_ok=True)

# Set IEEE formatting baselines
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['legend.fontsize'] = 9

# =====================================================================
# VISUALIZATION 1: FEATURE DISCUSSION VOLUME (AFR)
# =====================================================================
vol_pivot = agg_df.groupby(['Generation', 'Aspect']).size().unstack(fill_value=0).reindex(generations)
afr_pivot = vol_pivot.div(vol_pivot.sum(axis=1), axis=0) * 100

fig1, ax1 = plt.subplots(figsize=(6.5, 3.8), dpi=300)
markers = {'Performance': 'o', 'Camera': 's', 'Battery': '^', 'Display': 'D'}
colors = {'Performance': '#1f77b4', 'Camera': '#2ca02c', 'Battery': '#d62728', 'Display': '#ff7f0e'}
line_styles = {'Performance': '-', 'Camera': '--', 'Battery': '-.', 'Display': ':'}

for aspect in target_aspects:
    ax1.plot(x_ticks, afr_pivot[aspect], label=aspect, marker=markers[aspect], 
             color=colors[aspect], linestyle=line_styles[aspect], linewidth=1.8, markersize=5)

ax1.set_xlabel('Smartphone Generation ($G_n$)', labelpad=8)
ax1.set_ylabel('Aspect Frequency Ratio (AFR %)', labelpad=8)
ax1.grid(True, linestyle='--', alpha=0.4)
ax1.legend(loc='upper right', frameon=True, edgecolor='#e0e0e0')
plt.tight_layout()
plt.savefig("../data/plots/feature_volume_afr.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: Figure 1 -> ../data/plots/feature_volume_afr.png")

# =====================================================================
# VISUALIZATION 2: LONGITUDINAL SENTIMENT TRANSITIONS (2x2 SUBPLOTS)
# =====================================================================
fig2, axes = plt.subplots(2, 2, figsize=(7.2, 5.5), sharex=True, dpi=300)
axes = axes.flatten()

x_indices = np.arange(len(generations))
bar_width = 0.35

for i, aspect in enumerate(target_aspects):
    asp_df = agg_df[agg_df['Aspect'] == aspect]
    
    pos_counts, neg_counts = [], []
    for gen in generations:
        gen_df = asp_df[asp_df['Generation'] == gen]
        pos_counts.append(len(gen_df[gen_df['Sentiment'] == 'positive']))
        neg_counts.append(len(gen_df[gen_df['Sentiment'] == 'negative']))
        
    ax = axes[i]
    ax.bar(x_indices - bar_width/2, pos_counts, bar_width, label='Positive', color='#2ca02c', alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.bar(x_indices + bar_width/2, neg_counts, bar_width, label='Negative', color='#d62728', alpha=0.85, edgecolor='black', linewidth=0.5)
    
    ax.set_title(f'Domain Matrix: {aspect}', pad=6, weight='bold')
    ax.set_ylabel('Clause Volume (Count)')
    ax.set_xticks(x_indices)
    ax.set_xticklabels(x_ticks)
    ax.grid(True, linestyle='--', alpha=0.3, axis='y')
    
    if i == 0:
        ax.legend(loc='upper right', frameon=True, edgecolor='#e0e0e0')

plt.tight_layout()
plt.savefig("../data/plots/longitudinal_sentiment_subplots.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: Figure 2 -> ../data/plots/longitudinal_sentiment_subplots.png")

✅ Saved: Figure 1 -> ../data/plots/feature_volume_afr.png
✅ Saved: Figure 2 -> ../data/plots/longitudinal_sentiment_subplots.png
